<a href="https://colab.research.google.com/github/rltruter/Credit-Risk-Baseline/blob/main/02_feature_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount Drive and clone git


In [1]:
from google.colab import drive
drive.mount('/content/drive')
#Change filepath if different

Mounted at /content/drive


In [2]:
# If you're already in the repo directory in Colab, you can skip this cell.


# Option A: clone repo fresh each time (simple)
!git clone https://github.com/rltruter/Credit-Risk-Baseline.git



Cloning into 'Credit-Risk-Baseline'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 60 (delta 24), reused 36 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 44.19 KiB | 718.00 KiB/s, done.
Resolving deltas: 100% (24/24), done.


In [3]:
# Option B: if repo is already present, just cd into it
%cd /content/Credit-Risk-Baseline

/content/Credit-Risk-Baseline


# Load Config.py file

In [4]:
# Remove any conflicting /content/src directory that might be interfering with imports
!rm -rf /content/src

import os
import sys # Import sys
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Ensure the current working directory (project root) is at the beginning of sys.path
# This makes sure Python finds the correct 'src' module.
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Clear any cached 'src' or 'src.config' modules to force reload from the correct path
if 'src' in sys.modules:
    del sys.modules['src']
if 'src.config' in sys.modules:
    del sys.modules['src.config']

from src.config import (
    RAW_DATA_PATH, PROCESSED_DATA_DIR,
    BAD_STATUSES, GOOD_STATUSES, TARGET_COL,
    ISSUE_DATE_COL, ISSUE_DATE_FORMAT,
    LEAKAGE_COLS,
    DTI_COL, DTI_CAP,
    REV_UTIL_COL, REV_UTIL_CAP,
)

pd.set_option("display.max_columns", 200)

# Load Data

In [5]:
loan_data_df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print("Raw shape:", loan_data_df.shape)
loan_data_df[[ISSUE_DATE_COL, "loan_status"]].head()



Raw shape: (2260668, 145)


,issue_d,loan_status
0,Dec-2018,Current
1,Dec-2018,Current
2,Dec-2018,Current
3,Dec-2018,Current
4,Dec-2018,Current


# Recreate Target Variable

In [6]:
loan_data_df[TARGET_COL] = np.where(loan_data_df["loan_status"].isin(BAD_STATUSES), 1,
                          np.where(loan_data_df["loan_status"].isin(GOOD_STATUSES), 0, np.nan))

# Keep only loans with clear outcomes for supervised learning
loan_data_df = loan_data_df[loan_data_df[TARGET_COL].notna()].copy()
loan_data_df[TARGET_COL] = loan_data_df[TARGET_COL].astype(int)

loan_data_df[TARGET_COL].value_counts(normalize=True)


,proportion
target_default,
0,0.783852
1,0.216148


# Format Date Variable

In [7]:
loan_data_df[ISSUE_DATE_COL] = pd.to_datetime(loan_data_df[ISSUE_DATE_COL], format=ISSUE_DATE_FORMAT, errors="coerce")
missing_dates = loan_data_df[ISSUE_DATE_COL].isna().mean()
print("Missing issue_d fraction:", missing_dates)

loan_data_df = loan_data_df[loan_data_df[ISSUE_DATE_COL].notna()].copy()
loan_data_df["issue_year"] = loan_data_df[ISSUE_DATE_COL].dt.year
loan_data_df["issue_month"] = loan_data_df[ISSUE_DATE_COL].dt.to_period("M").astype(str)

loan_data_df[[ISSUE_DATE_COL, "issue_year", "issue_month"]].head()


Missing issue_d fraction: 0.0


,issue_d,issue_year,issue_month
100,2018-12-01,2018,2018-12
152,2018-12-01,2018,2018-12
170,2018-12-01,2018,2018-12
186,2018-12-01,2018,2018-12
215,2018-12-01,2018,2018-12


In [8]:
print("Unique years in loan_data_df:", sorted(loan_data_df["issue_year"].unique().tolist()))

Unique years in loan_data_df: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]


# Drop Leakage Columns

In [9]:
leakage_present = [c for c in LEAKAGE_COLS if c in loan_data_df.columns]
print("Leakage cols present:", len(leakage_present))

loan_data_df = loan_data_df.drop(columns=leakage_present)
print("Shape after leakage drop:", loan_data_df.shape)


Leakage cols present: 11
Shape after leakage drop: (1329272, 137)


# Feature Cleaning

In [10]:
# DTI cap
if DTI_COL in loan_data_df.columns:
    loan_data_df[DTI_COL] = pd.to_numeric(loan_data_df[DTI_COL], errors="coerce")
    loan_data_df["dti_capped"] = loan_data_df[DTI_COL].clip(lower=0, upper=DTI_CAP)

# Revolving utilization cap
if REV_UTIL_COL in loan_data_df.columns:
    loan_data_df[REV_UTIL_COL] = pd.to_numeric(loan_data_df[REV_UTIL_COL], errors="coerce")
    loan_data_df["revol_util_capped"] = loan_data_df[REV_UTIL_COL].clip(lower=0, upper=REV_UTIL_CAP)

loan_data_df[["dti_capped", "revol_util_capped"]].describe().T


,count,mean,std,min,25%,50%,75%,max
dti_capped,1328924.0,18.201063,8.650821,0.0,11.8,17.63,24.07,60.0
revol_util_capped,1328421.0,51.863002,24.475153,0.0,33.5,52.30,70.80,100.0


# Creating Candidate Dataset

In [11]:
# Drop columns that are identifiers or high-cardinality text
DROP_ALWAYS = [
    "id", "member_id", "url", "desc", "title",
    "zip_code", "emp_title",  # can be used later with NLP/high-cardinality handling
]

drop_present = [c for c in DROP_ALWAYS if c in loan_data_df.columns]
df_model = loan_data_df.drop(columns=drop_present).copy()

print("Candidate modeling shape:", df_model.shape)

Candidate modeling shape: (1329272, 132)


# Look for Missing Data

In [12]:
#Looking at which features have the highest proportion of missing values

missing = df_model.isnull().mean().sort_values(ascending=False)
missing.head(25)


,0
orig_projected_additional_accrued_interest,0.996160
sec_app_mths_since_last_major_derog,0.994772
payment_plan_start_date,0.994581
hardship_payoff_balance_amount,0.994581
deferral_term,0.994581
hardship_last_payment_amount,0.994581
hardship_dpd,0.994581
hardship_loan_status,0.994581
hardship_end_date,0.994581
hardship_start_date,0.994581


# Create Train and Validation Datasets

In [13]:
# Remove partial 2006 if it's sparse / messy
df_model = df_model[df_model["issue_year"] >= 2007].copy()

train_mask = (df_model["issue_year"] >= 2007) & (df_model["issue_year"] <= 2015)
val_mask   = (df_model["issue_year"] == 2016)
test_mask  = (df_model["issue_year"] >= 2017) & (df_model["issue_year"] <= 2018)

print("Train:", train_mask.sum(), "Val:", val_mask.sum(), "Test:", test_mask.sum())
print("Default rates:",
      df_model.loc[train_mask, TARGET_COL].mean(),
      df_model.loc[val_mask, TARGET_COL].mean(),
      df_model.loc[test_mask, TARGET_COL].mean())


Train: 825078 Val: 280472 Test: 223722
Default rates: 0.18655084731382973 0.25827890128069825 0.27248549539160205


# Save Training and Validation Artificats

In [14]:
df_train = df_model[train_mask].copy()
df_val   = df_model[val_mask].copy()
df_test  = df_model[test_mask].copy()

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
df_train.to_parquet(os.path.join(PROCESSED_DATA_DIR, "train.parquet"), index=False)
df_val.to_parquet(os.path.join(PROCESSED_DATA_DIR, "val.parquet"), index=False)
df_test.to_parquet(os.path.join(PROCESSED_DATA_DIR, "test.parquet"), index=False)
